# Day 41: Regularization - Ridge & Lasso

## The Problem: Overfitting
When a linear model has many features (especially more than the data can reliably support), plain least-squares regression tends to assign large, unstable coefficients that fit the training data very closely but generalize poorly - it's overfitting the noise, not just the signal. Regularization fights this by adding a penalty term to the loss function that grows with the size of the coefficients, so the optimizer is rewarded for keeping them small unless a feature earns its weight by genuinely reducing error. This produces simpler, more stable models that tend to perform better on unseen data. The code below builds a synthetic dataset where only two of the ten features (columns 0 and 1) actually matter, and the rest are pure noise - this setup lets us clearly see, later, whether Ridge/Lasso correctly downweight the useless features.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split

np.random.seed(0)
X = np.random.randn(100, 10)  # 100 samples, 10 features - only the first 2 will actually matter
y = X[:, 0] * 3 + X[:, 1] * 2 + np.random.randn(100) * 0.5  # true signal depends only on columns 0 and 1, plus noise
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Ridge Regression (L2)
Ridge adds an L2 penalty - the sum of squared coefficients - to the loss function, controlled by the `alpha` hyperparameter. This discourages any single coefficient from growing too large, which is especially useful when features are correlated (multicollinearity), since ordinary linear regression tends to produce wild, unstable coefficients in that situation. Ridge shrinks all coefficients toward zero proportionally but almost never sets them exactly to zero, so it keeps every feature in the model, just with reduced influence. When you read the output below, expect the coefficients for features 0 and 1 (the real signal) to stay relatively large while the other 8 (pure noise) shrink toward small values, and R2 tells you how much of the test-set variance in y the model explains (closer to 1 is better).

In [ ]:
ridge = Ridge(alpha=1.0).fit(X_train, y_train)  # alpha=1.0 is a moderate regularization strength
print('Ridge coefficients:', ridge.coef_.round(2))  # coefficients for features 0 and 1 should dominate; others shrink toward 0
print('Ridge R2:', ridge.score(X_test, y_test))  # R2 on held-out test data - measures generalization, not just training fit

## Lasso Regression (L1)
Lasso uses an L1 penalty - the sum of the absolute values of the coefficients - instead of Ridge's squared penalty. This subtle change in the math has a big practical effect: L1 regularization can drive coefficients all the way to exactly zero, not just shrink them close to it. That means Lasso effectively performs automatic feature selection, dropping features it decides aren't useful. This is valuable when you suspect many of your features are irrelevant and you want a sparser, more interpretable model. In the output below, watch for coefficients that become exactly 0.0 - those are the features Lasso has eliminated, ideally the noise columns (2 through 9) rather than the real signal columns (0 and 1).

In [ ]:
lasso = Lasso(alpha=0.1).fit(X_train, y_train)  # alpha=0.1: smaller than Ridge's alpha since L1 penalty behaves differently in scale
print('Lasso coefficients:', lasso.coef_.round(2))  # look for exact zeros - those features were dropped entirely
print('Lasso R2:', lasso.score(X_test, y_test))

## Choosing Alpha
`alpha` is the knob that controls how strongly the model is regularized, and picking it well is the whole game with Ridge/Lasso. When alpha is very small, the penalty barely matters and the model behaves almost like plain linear regression (risk of overfitting); when alpha is very large, the penalty dominates and coefficients get crushed toward zero regardless of how useful the features are (risk of underfitting). In practice alpha is chosen via cross-validation on a range of values (or using `RidgeCV`/`LassoCV`), not by eyeballing a handful of numbers. The loop below sweeps several alpha values and prints the resulting test R2 for each, so you can see the sweet spot where performance peaks before it starts declining again as alpha grows too large.

In [ ]:
for alpha in [0.01, 0.1, 1, 10]:  # sweep from light to heavy regularization
    r = Ridge(alpha=alpha).fit(X_train, y_train)
    print(f'alpha={alpha}: R2={r.score(X_test, y_test):.3f}')  # watch how R2 changes as regularization strength increases